# Explore EIA/FERC utility entity matching with Splink

This notebook loads the FERC and EIA utility tables from a local copy plopped into `devtools/`, inspects the available columns, builds a normalized set of candidate match fields, and uses Splink to explore how the records line up.

This is just a draft, obvious improvements are:
- reading in the pickles from dagster storage or (eventually) reading in the parquet files
- more robust match-specific cleaning
- ideally we'd like `utility_id_ferc1` and `report_year` to be a unique key for the FERC dataframe. right now it's not, which means we're doing the match with a lot of near duplicate records or records that could be consolidated by filling nulls, which isn't great.

The main goals are:
- inspect the available columns in both tables
- focus on `name`, `year`, `street_address`, `city`, `state`, and `zip`
- surface other possible identifiers that might help matching
- compare the two FERC address columns to see which one is more useful for linkage
- generate candidate utility matches with Splink for manual review

Next:
- fix the exact match cell issue
- clean utility names
- try running match without address


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from dagster import AssetKey
from pudl.etl import defs
from pudl.analysis.record_linkage.name_cleaner import CompanyNameCleaner

import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
from splink.blocking_analysis import count_comparisons_from_blocking_rule
from splink.exploratory import completeness_chart, profile_columns

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

In [ ]:
eia_raw = pd.read_parquet("s3://pudl.catalyst.coop/nightly/out_eia__yearly_utilities.parquet")

In [ ]:
ferc_raw = pd.read_parquet("s3://pudl.catalyst.coop/nightly/core_ferc1__yearly_identification_certification.parquet")

In [ ]:
ferc_raw = defs.load_asset_value(AssetKey("core_ferc1__yearly_identification_certification"))
eia_raw = defs.load_asset_value(AssetKey("out_eia__yearly_utilities"))

In [ ]:
ferc_raw.shape, eia_raw.shape

Read in training data (manually validated matches)

In [ ]:
labels_df = pd.read_csv("utility_match_training_data.csv", dtype = {"utility_id_pudl": str,
                                                                      "utility_id_ferc1": str,
                                                                      "utility_id_eia": str,
                                                                      "utility_name_ferc1": str,
                                                                      "utility_name_eia": str,
                                                                     })

The manually validated matches are the rows with a non-null address_match column.

In [ ]:
labels_df.head(5)

In [ ]:
labels_df.address_match.isnull().value_counts()

Let's see what potential columns are available on each side to match on.

In [ ]:
COLUMN_GROUPS = {
    "name", "date", "year", "address", "city", "zip", "state", "id"
}

def get_cols_in_group(df, group_str):
    return [col for col in df if group_str in col]

candidate_cols = {}
for group in COLUMN_GROUPS:
    candidate_cols[group] = {}
    candidate_cols[group]["ferc"] = get_cols_in_group(ferc_raw, group)
    candidate_cols[group]["eia"] = get_cols_in_group(eia_raw, group)

In [ ]:
pd.DataFrame(candidate_cols)

# Investigate candidate match columns

Now we can narrow down the columns of interest and take a look at some of the candidate match columns. We want to check for their nullness, the uniqueness of values, any obvious cleaning needs, etc.

In [ ]:
ferc_candidate_cols = [
    "utility_id_ferc1",
    "office_state",
    "contact_state",
    "utility_name_ferc1",
    "contact_name",
    "office_zip",
    "contact_zip",
    "filing_date",
    "report_year",
    "office_street_address",
    "contact_address",
    "office_city",
    "contact_city"
]
eia_candidate_cols = [
    "utility_id_eia",
    "state",
    "utility_name_eia",
    "contact_firstname",
    "contact_lastname",
    "contact_firstname_2",
    "contact_lastname_2",
    "report_date",
    "zip_code",
    "street_address",
    "address_2",
    "city"
]

In [ ]:
ferc_raw[ferc_candidate_cols].info()

In [ ]:
eia_raw[eia_candidate_cols].info()

In [ ]:
ferc_raw[ferc_candidate_cols].isnull().sum()/len(ferc_raw)

In [ ]:
eia_raw[eia_candidate_cols].isnull().sum()/len(eia_raw)

Ok we can already tell that contact name won't be good to match on since there's lots of nulls in EIA. So far it seems like the following will be good:
- utility name
- address (likely office but maybe contact address if it matches better with EIA address)
- zip code (same question of office vs contact)
- city

We'll use report date / year for something called blocking, which means only record pairs with matching report years will be considered.

In [ ]:
eia_candidate_cols = [col for col in eia_candidate_cols if col not in ["contact_firstname", "contact_lastname", "contact_firstname_2", "contact_lastname_2"]]
ferc_candidate_cols = [col for col in ferc_candidate_cols if col not in ["contact_name", "filing_date"]]

## Profile Columns

We have an idea of nullness, let's learn more about the candidate columns by analyzing the distribution of values in the columns using a built in splink functionality. This is important because (taken from [splink docs](https://moj-analytical-services.github.io/splink/demos/tutorials/02_Exploratory_analysis.html)):

```
The distribution of values in your data is important for two main reasons:

Columns with higher cardinality (number of distinct values) are usually more useful for data linking. For instance, date of birth is a much stronger linkage variable than gender.

The skew of values is important. If you have a city column that has 1,000 distinct values, but 75% of them are London, this is much less useful for linkage than if the 1,000 values were equally distributed
```

In [ ]:
eia_profile_cols = [col for col in eia_candidate_cols if col not in ["utility_id_eia", "report_date"]]

profile_columns(eia_raw[eia_profile_cols], db_api=DuckDBAPI(), top_n=10, bottom_n=5)

In [ ]:
ferc_profile_cols = [col for col in ferc_candidate_cols if col not in ["utility_id_ferc1", "report_year"]]

profile_columns(ferc_raw[ferc_profile_cols], db_api=DuckDBAPI(), top_n=10, bottom_n=5)

There's definitely some weird skew with the most sampled values on addresses, cities, and zips on both the EIA and FERC side... but nothing too worrying at this point.

# Select match columns and rename

Lets take an initial stab at selecting some columns for matching. For now I'm going to go with the office address and city over the contact address and city.

In [ ]:
match_cols = {
    "utility_name": {"eia": "utility_name_eia", "ferc": "utility_name_ferc1"},
    "street_address": {"eia": "street_address", "ferc": "office_street_address"},
    "city": {"eia": "city", "ferc": "office_city"},
    "zip": {"eia": "zip_code", "ferc": "office_zip"},
    "state": {"eia": "state", "ferc": "office_state"}
}

In [ ]:
eia_df = eia_raw.copy()
ferc_df = ferc_raw.copy()
for col, rename_dict in match_cols.items():
    eia_df = eia_df.rename(columns={rename_dict["eia"]: col})
    ferc_df = ferc_df.rename(columns={rename_dict["ferc"]: col})

In [ ]:
eia_df["report_year"] = pd.to_datetime(eia_df["report_date"]).dt.year

# Clean for matching

Let's do some basic cleaning to get the dataframes ready for matching. Splink suggests:

* unique IDs for records
* standardizing date formats, matching text case, and handling invalid data
* Ensure null values (or other 'not known' indicators) are represented as true nulls, not empty strings

In [ ]:
eia_df[["utility_id_eia", "report_year"]].duplicated().value_counts()

In [ ]:
ferc_df[["utility_id_ferc1", "report_year"]].duplicated().value_counts()

In [ ]:
ferc_df[ferc_df[["utility_id_ferc1", "report_year"]].duplicated(keep=False)].sort_values(by=["utility_id_ferc1", "report_year"])

This 1 duplicated record seems okay to ignore. We know roughly that utility_id_eia and report_year form a unique key for the datasets, so we can be somewhat reassured that if we use report_year as a blocking key, then there won't be a ton of near duplicates on each side that we're trying to match.

Perform some basic cleaning

In [ ]:
match_cols

In [ ]:
def clean_for_matching(df, dataset_name: str):
    df["source_dataset"] = dataset_name
    df["record_id"] = df.index.astype("string")
    for str_col in ["utility_name", "street_address", "city", "zip", "state"]:
        df[str_col] = df[str_col].astype("string").str.strip().str.lower().replace("", pd.NA)
    return df

In [ ]:
ferc_df = clean_for_matching(ferc_df, "ferc")

In [ ]:
eia_df = clean_for_matching(eia_df, "eia")

Use CompanyNameCleaner to clean utility name. Along with some more string cleaning (removing word "the", removing special chars, etc) this will expand legal terms, i.e. "llc" to "limited liabilty company". You have the option to remove the legal terms entirely, which could be a good option here, by passing in `handle_legal_terms=2`.

In [ ]:
# we elect to remove legal terms
name_cleaner = CompanyNameCleaner(handle_legal_terms=2)

In [ ]:
ferc_df["utility_name"] = name_cleaner.apply_name_cleaning(ferc_df["utility_name"])

In [ ]:
eia_df["utility_name"] = name_cleaner.apply_name_cleaning(eia_df["utility_name"])

In [ ]:
eia_df["utility_name"].head(5)

## Side quest to verify that office address is better than contact address on the ferc side

In [ ]:
ferc_df_contact_address = ferc_df.copy()
ferc_df_contact_address["street_address"] = ferc_df_contact_address["contact_address"].astype("string").str.strip().str.lower().replace("", pd.NA)

In [ ]:
def address_option_summary(ferc_df: pd.DataFrame, eia_df: pd.DataFrame, street_col: str) -> dict[str, object]:
    ferc_non_null = ferc_df[ferc_df["street_address"].notna()].copy()
    eia_non_null = eia_df[eia_df["street_address"].notna()].copy()

    full_address_keys = ["report_year", "street_address", "city", "state", "zip"]
    loose_address_keys = ["street_address", "city", "state"]
    name_state_keys = ["report_year", "utility_name", "state"]

    exact_full = ferc_non_null[full_address_keys].merge(
        eia_non_null[full_address_keys].drop_duplicates(),
        how="inner",
        on=full_address_keys,
    )
    exact_loose = ferc_non_null[loose_address_keys].merge(
        eia_non_null[loose_address_keys].drop_duplicates(),
        how="inner",
        on=loose_address_keys,
    )
    name_state = ferc_df[name_state_keys].merge(
        eia_df[name_state_keys].drop_duplicates(),
        how="inner",
        on=name_state_keys,
    )

    return {
        "ferc_street_column": street_col,
        "ferc_records": len(ferc_df),
        "ferc_non_null_street": int(ferc_df["street_address"].notna().sum()),
        "ferc_street_fill_pct": round(100 * ferc_df["street_address"].notna().mean(), 1),
        "ferc_unique_streets": int(ferc_df["street_address"].nunique(dropna=True)),
        "exact_full_address_overlap": len(exact_full),
        "exact_loose_address_overlap": len(exact_loose),
        "name_state_overlap": len(name_state),
    }


ferc_match_frames = {
    "office_address": ferc_df,
    "contact_address": ferc_df_contact_address
}

address_option_df = pd.DataFrame(
    [
        address_option_summary(ferc_df, eia_df, street_col)
        for street_col, ferc_df in ferc_match_frames.items()
    ]
).sort_values(
    ["exact_full_address_overlap", "exact_loose_address_overlap", "ferc_street_fill_pct"],
    ascending=False,
)

display(address_option_df)


Okay, office_address is the better address to go with, as we suspected.

# Before splinking, check for exact matches with a merge and see what we can learn

In [ ]:
# first just merge on report year, utility name, and street address
name_street_exact_candidates = ferc_df.merge(
        eia_df,
        how="inner",
        on=["report_year", "utility_name", "street_address"],
        suffixes=("_ferc", "_eia"),
)

In [ ]:
len(name_street_exact_candidates)

In [ ]:
len(name_street_exact_candidates.utility_id_ferc1.unique())/len(ferc_df.utility_id_ferc1.unique())

In [ ]:
# try merging on report_year, state, and utility name and see what we can learn about agreement between other matching columns
quick_exact_candidates = (
    ferc_df.merge(
        eia_df,
        how="inner",
        on=["report_year", "state", "utility_name"],
        suffixes=("_ferc", "_eia"),
    )
    .assign(
        same_street=lambda df: df["street_address_ferc"].fillna("").eq(df["street_address_eia"].fillna("")),
        same_city=lambda df: df["city_ferc"].fillna("").eq(df["city_eia"].fillna("")),
        same_zip=lambda df: df["zip_ferc"].fillna("").eq(df["zip_eia"].fillna("")),
    )
)

quick_exact_candidates["agreement_score"] = quick_exact_candidates[
    ["same_street", "same_city", "same_zip"]
].sum(axis=1)

quick_exact_candidates = quick_exact_candidates.sort_values(
    ["agreement_score", "same_street", "same_zip"],
    ascending=False,
)


quick_exact_candidates[
    [
            "record_id_ferc",
            "utility_name",
            "street_address_ferc",
            "city_ferc",
            "state",
            "zip_ferc",
            "record_id_eia",
            "street_address_eia",
            "city_eia",
            "zip_eia",
            "agreement_score",
            "same_street",
            "same_city",
            "same_zip",
    ]
].head(10)

In [ ]:
quick_exact_candidates.agreement_score.value_counts()

# Splink it!

We start by prepping the labeled set and defining these comparison rules/thresholds for each match column. We also define "blocking rules" which say which pairs we should even consider as potential matches. Let's start by testing how big of a "candidate pair set" we create with each proposed blocking rule.

In [ ]:
match_cols.keys()

In [ ]:
cols_to_keep = list(match_cols.keys()) + [
    "record_id",
    "report_year",
    "source_dataset",
    "utility_id_eia",
    "utility_id_ferc1",
]

eia_match_df = eia_df[[col for col in cols_to_keep if col in eia_df.columns]].copy()
ferc_match_df = ferc_df[[col for col in cols_to_keep if col in ferc_df.columns]].copy()

# Splink vertically concatenates input tables, so they need matching schemas.
eia_match_df["utility_id_eia"] = eia_match_df["utility_id_eia"].astype("Int64")
eia_match_df["utility_id_ferc1"] = pd.Series(pd.NA, index=eia_match_df.index, dtype="Int64")
ferc_match_df["utility_id_eia"] = pd.Series(pd.NA, index=ferc_match_df.index, dtype="Int64")
ferc_match_df["utility_id_ferc1"] = ferc_match_df["utility_id_ferc1"].astype("Int64")

eia_match_df = eia_match_df[cols_to_keep]
ferc_match_df = ferc_match_df[cols_to_keep]

## Prepare the labeled set

We're going to use the manually verified utility matches as training data (or labels) for the match. Prepare that data into the format splink expects. There's a tutorial on linking with labels in Splink [here](https://moj-analytical-services.github.io/splink/demos/examples/duckdb/pairwise_labels.html).

Since the labeled data doesn't have a time axis, we're going to find the most recent `report_year` shared by each labeled `utility_id_ferc1` / `utility_id_eia` pair. Then we'll use the records from that shared year to create the `source_dataset_l`, `record_id_l`, `source_dataset_r`, and `record_id_r` columns Splink expects for pairwise labels.

In [ ]:
labels_df[["utility_id_ferc1", "utility_id_eia"]].duplicated().value_counts()

In [ ]:
# let's start with just the validated pairs
validated_labels_df = labels_df.loc[labels_df["address_match"].notna()].copy()
validated_labels_df = validated_labels_df.reset_index(names="label_row_id")

In [ ]:
# get the most recent record for each FERC and EIA utility
ferc_record_lookup = (
    ferc_match_df.loc[
        ferc_match_df["utility_id_ferc1"].notna(),
        ["utility_id_ferc1", "report_year", "record_id"],
    ]
    .sort_values(["utility_id_ferc1", "report_year"], ascending=[True, False])
    .drop_duplicates(["utility_id_ferc1", "report_year"], keep="first")
    .rename(columns={"record_id": "record_id_r"})
)

eia_record_lookup = (
    eia_match_df.loc[
        eia_match_df["utility_id_eia"].notna(),
        ["utility_id_eia", "report_year", "record_id"],
    ]
    .sort_values(["utility_id_eia", "report_year"], ascending=[True, False])
    .drop_duplicates(["utility_id_eia", "report_year"], keep="first")
    .rename(columns={"record_id": "record_id_l"})
)

Splink wants this clerical match score column. Since these are manually validated, it's 1 for all rows.

In [ ]:
validated_labels_df["clerical_match_score"] = 1

In [ ]:
# need to make these ints for merging, any reason not to?
for id_col in ["utility_id_pudl", "utility_id_ferc1", "utility_id_eia"]:
    validated_labels_df[id_col] = pd.to_numeric(
        validated_labels_df[id_col], errors="coerce"
    ).astype("Int64")

In [ ]:
# we want to find the records with the latest shared report_year
# for each labeled utility_id_ferc1 / utility_id_eia pair in the training data
label_record_candidates = (
    validated_labels_df.merge(ferc_record_lookup, on="utility_id_ferc1", how="left")
    .merge(eia_record_lookup, on=["utility_id_eia", "report_year"], how="inner")
    .sort_values(["label_row_id", "report_year"], ascending=[True, False])
)

labeled_df = (
    label_record_candidates.drop_duplicates("label_row_id", keep="first")
    .assign(source_dataset_l="eia", source_dataset_r="ferc")
    [
        [
            "source_dataset_l",
            "record_id_l",
            "source_dataset_r",
            "record_id_r",
            "clerical_match_score",
            "report_year",
            "utility_id_pudl",
            "utility_id_ferc1",
            "utility_name_ferc1",
            "utility_id_eia",
            "utility_name_eia",
            "address_match",
        ]
    ]
)

labeled_df.head()

In [ ]:
missing_labeled_pairs = validated_labels_df.loc[
    ~validated_labels_df["label_row_id"].isin(label_record_candidates["label_row_id"]),
    ["utility_id_ferc1", "utility_name_ferc1", "utility_id_eia", "utility_name_eia", "address_match"],
]

In [ ]:
print(f"Validated labels: {len(validated_labels_df)}")
print(f"Labels with a shared report_year record pair: {len(labeled_df)}")
print(f"Labels without a shared report_year record pair: {len(missing_labeled_pairs)}")
display(missing_labeled_pairs.head(20))

## Set model parameters

Note on the "probability two random record match" parameter:
- this can be estimated using random sampling
- however in the splink docs it reads: "if you are linking two tables of 10,000 records and expect a one-to-one match, then you should set this value to 1/10_000 in your settings instead of estimating it."
- we don't exactly have a 1 to 1 match, but I think it's probably close enough to just use 1 over the EIA utilities table length

Estimating m parameter:
- if we're using labels / training data, then we estimate m differently than if we're doing this unsupervised (using expectation maximization)
- see commented out line below

Things to play with when changing comparisons:
- the distance metric (I find jaro winkler works the best but have had success with Jaccard)
- creating a metaphone column for utility name (see the splink-ferc-eia-match) and passing in `dmeta_col_name` for the NameComparison
- the number of thresholds in the street address comparison. I used fewer here because I don't want to over-emphasize the importance of street address compared to utility name, I kind of care the most if it's an exact match or a near-match, but below that I don't care that much.

In [ ]:
comparison_name = cl.NameComparison(
    "utility_name",
    jaro_winkler_thresholds=[0.97, 0.93, 0.88],
)
comparison_street = cl.NameComparison(
    "street_address",
    jaro_winkler_thresholds=[0.97],
)
comparison_city = cl.NameComparison(
    "city",
    # jaro_winkler_thresholds=[0.95, 0.88],
    jaro_winkler_thresholds=[0.95],
)
comparison_state = cl.ExactMatch("state")
comparison_zip = cl.ExactMatch("zip")
comparison_year = cl.ExactMatch("report_year")

# these term frequency adjustments make it so commonly occurring states/zips are weighted less
comparison_state.configure(term_frequency_adjustments=True)
comparison_zip.configure(term_frequency_adjustments=True)

blocking_rules = [
    block_on("report_year", "state"),
    block_on("report_year", "zip"),
]

for blocking_rule in blocking_rules:
    display(Markdown(f"### Blocking rule\n`{blocking_rule}`"))
    display(
        count_comparisons_from_blocking_rule(
            table_or_tables=[eia_match_df, ferc_match_df],
            blocking_rule=blocking_rule,
            link_type="link_only",
            unique_id_column_name="record_id",
            db_api=DuckDBAPI(),
        )
    )


In [ ]:
settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="record_id",
    comparisons=[
        comparison_name,
        comparison_year,
        comparison_street,
        comparison_city,
        comparison_state,
        comparison_zip,
    ],
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
    probability_two_random_records_match=1 / len(eia_match_df),
)

linker = Linker(
    [eia_match_df, ferc_match_df],
    settings=settings,
    input_table_aliases=["eia", "ferc"],
    db_api=DuckDBAPI(),
)

linker.training.estimate_u_using_random_sampling(max_pairs=1e6)
# register the labels with the linker
labels_df = linker.table_management.register_labels_table(labeled_df, overwrite=True)
linker.training.estimate_m_from_pairwise_labels(labels_df)

If we're not using labels, then m can be estimated using an unsupervised method. Let's also do that and compare the supervised and unsupervised training sessions with the below visuals.

In [ ]:
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("report_year", "state")
)

## Look at some visuals to see if the model parameters look good before making predictions

This chart compares m values from different training sessions. The EM (blue circles) is unsupervised, the orange squares the labeled data training run. The m values are the proportion of records falling into each ComparisonLevel amongst truly matching records, so I think it makes sense that the unsupervised run is more pessimistic, because it's assuming that "weird matches" are not truly matching records. Maybe that's a half baked thought, this chart is a little hard to infer anything from. Ignore report_year here as we're only using it for blockign.

In [ ]:
linker.visualisations.parameter_estimate_comparisons_chart()

This chart indicates how the linker will assign a match weight to the pair based on if that pair falls into each comparison level. We want comparison levels that we think are good indicators of a match (like exact match on address) to be weighted highly. Ignore report_year here.

In [ ]:
linker.visualisations.match_weights_chart()

Let's see what comparison levels the true matching records (according to the model) fall into, and which comparison levels the non-matching records fall into (hopefully this is "All other comparisons"). 

In [ ]:
linker.visualisations.m_u_parameters_chart()

I think we'd like to see more records falling into exact match (or close match) on address and utility name. This could be something to return to when tuning the model.

In [ ]:
predictions = linker.inference.predict(threshold_match_probability=0.1)
preds_df = predictions.as_pandas_dataframe().sort_values("match_probability", ascending=False)

candidate_review = preds_df.merge(
    eia_match_df.add_suffix("_eia"),
    left_on="record_id_l",
    right_on="record_id_eia",
    how="left",
).merge(
    ferc_match_df.add_suffix("_ferc"),
    left_on="record_id_r",
    right_on="record_id_ferc",
    how="left",
)

review_columns = [
    "match_probability",
    "record_id_ferc",
    "utility_id_ferc1_ferc",
    "utility_name_ferc",
    "street_address_ferc",
    "city_ferc",
    "state_ferc",
    "zip_ferc",
    "record_id_eia",
    "utility_id_eia_eia",
    "utility_name_eia",
    "street_address_eia",
    "city_eia",
    "state_eia",
    "zip_eia",
    "gamma_utility_name",
    "gamma_street_address",
    "gamma_city",
    "gamma_state",
    "gamma_zip",
    "gamma_report_year",
]

display(candidate_review[review_columns].head(50))

In [ ]:
preds = candidate_review[candidate_review.match_probability > .99]

In [ ]:
less_than_one = preds[(preds.match_probability<.9999) & (preds.match_probability>.995)]

In [ ]:
less_than_one_review_columns = [
    "match_probability",
    "record_id_ferc",
    "utility_id_ferc1_ferc",
    "utility_name_ferc",
    "street_address_ferc",
    "city_ferc",
    "state_ferc",
    "zip_ferc",
    "record_id_eia",
    "utility_id_eia_eia",
    "utility_name_eia",
    "street_address_eia",
    "city_eia",
    "state_eia",
    "zip_eia",]

less_than_one[less_than_one_review_columns]

Some of these look better, some still look bad. Let's take a look at some of the bad ones and see why they're being matched.

In [ ]:
waterfall_records = less_than_one.head(3)[preds_df.columns].to_dict("records")
linker.visualisations.waterfall_chart(waterfall_records)

Based on the above, it seems like a matching zip (and maybe city) is being weighted too highly, or that the correlation between zip and city is too high. If a pair has matching zip then it likely has matching city, so we're effectively doubling the weight given to this one shared locational attribute. Next: Let's try running the model without city.

Splink also has a comparison viewer dashboard and a clustering dashboard which seem helpful but I haven't explored them.

## Try a model without city comparison

In [ ]:
settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="record_id",
    comparisons=[
        comparison_name,
        comparison_year,
        comparison_street,
        # comparison_city,
        comparison_state,
        comparison_zip,
    ],
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
    probability_two_random_records_match=1 / len(eia_match_df),
)

linker = Linker(
    [eia_match_df, ferc_match_df],
    settings=settings,
    input_table_aliases=["eia", "ferc"],
    db_api=DuckDBAPI(),
)

linker.training.estimate_u_using_random_sampling(max_pairs=1e6)
# register the labels with the linker
labels_df = linker.table_management.register_labels_table(labeled_df, overwrite=True)
linker.training.estimate_m_from_pairwise_labels(labels_df)

In [ ]:
linker.visualisations.match_weights_chart()

In [ ]:
linker.visualisations.m_u_parameters_chart()

In [ ]:
predictions = linker.inference.predict(threshold_match_probability=0.1)
preds_df = predictions.as_pandas_dataframe().sort_values("match_probability", ascending=False)

candidate_review = preds_df.merge(
    eia_match_df.add_suffix("_eia"),
    left_on="record_id_l",
    right_on="record_id_eia",
    how="left",
).merge(
    ferc_match_df.add_suffix("_ferc"),
    left_on="record_id_r",
    right_on="record_id_ferc",
    how="left",
)

review_columns = [
    "match_probability",
    "record_id_ferc",
    "utility_id_ferc1_ferc",
    "utility_name_ferc",
    "street_address_ferc",
    "city_ferc",
    "state_ferc",
    "zip_ferc",
    "record_id_eia",
    "utility_id_eia_eia",
    "utility_name_eia",
    "street_address_eia",
    "city_eia",
    "state_eia",
    "zip_eia",
    "gamma_utility_name",
    "gamma_street_address",
    "gamma_state",
    "gamma_zip",
    "gamma_report_year",
]

display(candidate_review[review_columns].head(50))

In [ ]:
preds = candidate_review[candidate_review.match_probability > .99]

In [ ]:
less_than_one = preds[(preds.match_probability<.9999) & (preds.match_probability>.995)]

In [ ]:
less_than_one_review_columns = [
    "match_probability",
    "record_id_ferc",
    "utility_id_ferc1_ferc",
    "utility_name_ferc",
    "street_address_ferc",
    "city_ferc",
    "state_ferc",
    "zip_ferc",
    "record_id_eia",
    "utility_id_eia_eia",
    "utility_name_eia",
    "street_address_eia",
    "city_eia",
    "state_eia",
    "zip_eia",]

less_than_one[less_than_one_review_columns].tail(50)

Some of these look better, some still look bad. Let's take a look at some of the bad ones and see why they're being matched.

In [ ]:
waterfall_records = less_than_one.tail(3)[preds_df.columns].to_dict("records")
linker.visualisations.waterfall_chart(waterfall_records)

In [ ]:
candidate_review[candidate_review.match_probability >= .9].tail(25)

In [ ]:
out = candidate_review[candidate_review.match_probability > .9]

In [ ]:
out

Overall it seems like this model with city removed works better. Manually review the results to see how good a job it's doing.

In [ ]:
len(out.utility_id_ferc1_ferc.unique()), len(out.utility_id_eia_eia.unique())

In [ ]:
len(out.utility_id_ferc1_ferc.unique())/len(ferc_raw.utility_id_ferc1.unique()), len(out.utility_id_eia_eia.unique())/len(eia_raw.utility_id_eia.unique())

In [ ]:
out.to_parquet("full_matched_output.parquet")

In [ ]:
best_out = (
    out.sort_values(
        ["utility_id_ferc1_ferc", "match_probability", "utility_id_eia_eia"],
        ascending=[True, False, True],
    )
    .drop_duplicates(subset=["utility_id_ferc1_ferc"], keep="first")
    .reset_index(drop=True)
)

In [ ]:
best_out.to_parquet("assn_table_output.parquet")

Let's see how predictions look compared to the labels. Remember that 4 of the pairs in the labeled set weren't even part of our candidate match set (because we couldn't get a matching report years for the utility pair).

This chart shows some metrics about the predictions. Ignore precision here, because we only have 1 label (a match) in our labeled set, the model will always be correct when it makes a prediction on a pair from the labeled set. What we're battling with is recall, and it's currently not getting amazing recall. Recall really falls off after .9, which could indicate a good match threshold.

In [ ]:
linker.evaluation.accuracy_analysis_from_labels_table(
    labels_df,
    output_type="threshold_selection",
    add_metrics=["accuracy", "f1", "f0_5", "f2", "specificity", "npv", "phi"],
)

In [ ]:
label_eval_table = linker.evaluation.accuracy_analysis_from_labels_table(
    labels_df,
    threshold_match_probability=0.5,
    output_type="table",
    add_metrics=["accuracy", "f1", "f0_5", "f2", "specificity", "npv", "phi"],
).as_pandas_dataframe()

metrics_at_09 = label_eval_table.loc[
    label_eval_table["match_probability"] >= 0.9
].sort_values("match_probability").head(1)

if metrics_at_09.empty:
    metrics_at_09 = label_eval_table.iloc[
        (label_eval_table["match_probability"] - 0.9).abs().argsort()[:1]
    ]

metrics_cols = [
    "match_probability",
    "tp",
    "fp",
    "tn",
    "fn",
    "precision",
    "recall",
    "specificity",
    "npv",
    "accuracy",
    "f1",
    "f0_5",
    "f2",
    "phi",
]

metrics_at_09[metrics_cols]

These are the matches that we miss from the labeled set with a threshold match probability of .9

In [ ]:
prediction_errors_at_09 = linker.evaluation.prediction_errors_from_labels_table(
    labels_df,
    threshold_match_probability=0.9,
).as_pandas_dataframe()

prediction_errors_at_09

## Evaluate only the top match per FERC utility

Splink's built-in evaluation charts work on pairwise predictions. If we force a single best match per `utility_id_ferc1`, that's a post-processing step, so we evaluate that reduced result set manually against the labeled pairs.

In [ ]:
best_out[
    [
        "utility_id_ferc1_ferc",
        "utility_name_ferc",
        "utility_id_eia_eia",
        "utility_name_eia",
        "match_probability",
    ]
].sort_values(by="match_probability", ascending=False)

Hmmm I think there's mahybe bug in the below cells...

In [ ]:
best_out_for_eval = best_out.rename(
    columns={
        "utility_id_ferc1_ferc": "utility_id_ferc1",
        "utility_id_eia_eia": "predicted_utility_id_eia",
        "utility_name_eia": "predicted_utility_name_eia",
        "match_probability": "predicted_match_probability",
    }
)

best_match_label_eval = (
    labeled_df[
        [
            "utility_id_pudl",
            "utility_id_ferc1",
            "utility_name_ferc1",
            "utility_id_eia",
            "utility_name_eia",
            "address_match",
            "clerical_match_score",
        ]
    ]
    .merge(
        best_out_for_eval[
            [
                "utility_id_ferc1",
                "predicted_utility_id_eia",
                "predicted_utility_name_eia",
                "predicted_match_probability",
            ]
        ],
        on="utility_id_ferc1",
        how="left",
    )
)

best_match_label_eval["actual_match"] = best_match_label_eval["clerical_match_score"].eq(1)
best_match_label_eval["predicted_match"] = best_match_label_eval["utility_id_eia"].eq(
    best_match_label_eval["predicted_utility_id_eia"]
)
best_match_label_eval["predicted_match"] = best_match_label_eval["predicted_match"].fillna(False)
# add label for true neg, true pos, false pos, false neg
best_match_label_eval["validation_bucket"] = "tn"
best_match_label_eval.loc[
    best_match_label_eval["actual_match"] & best_match_label_eval["predicted_match"],
    "validation_bucket",
] = "tp"
best_match_label_eval.loc[
    ~best_match_label_eval["actual_match"] & best_match_label_eval["predicted_match"],
    "validation_bucket",
] = "fp"
best_match_label_eval.loc[
    best_match_label_eval["actual_match"] & ~best_match_label_eval["predicted_match"],
    "validation_bucket",
] = "fn"

best_match_label_eval


In [ ]:
confusion_counts = best_match_label_eval["validation_bucket"].value_counts()

tp = int(confusion_counts.get("tp", 0))
fp = int(confusion_counts.get("fp", 0))
tn = int(confusion_counts.get("tn", 0))
fn = int(confusion_counts.get("fn", 0))

precision = tp / (tp + fp) if (tp + fp) else pd.NA
recall = tp / (tp + fn) if (tp + fn) else pd.NA
specificity = tn / (tn + fp) if (tn + fp) else pd.NA
npv = tn / (tn + fn) if (tn + fn) else pd.NA
accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else pd.NA
f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else pd.NA

best_match_metrics = pd.DataFrame(
    {
        "metric": [
            "tp",
            "fp",
            "tn",
            "fn",
            "precision",
            "recall",
            "specificity",
            "npv",
            "accuracy",
            "f1",
        ],
        "value": [tp, fp, tn, fn, precision, recall, specificity, npv, accuracy, f1],
    }
)

best_match_metrics


In [ ]:
best_match_errors = best_match_label_eval.loc[
    best_match_label_eval["validation_bucket"].isin(["fp", "fn"])
].sort_values(["validation_bucket", "utility_id_ferc1", "utility_id_eia"])

best_match_errors[
    [
        "validation_bucket",
        "utility_id_ferc1",
        "utility_name_ferc1",
        "utility_id_eia",
        "utility_name_eia",
        "predicted_utility_id_eia",
        "predicted_utility_name_eia",
        "predicted_match_probability",
        "address_match",
    ]
]


## Next steps

**Cleaning**
- clean company names with name cleaner
- try a run with no address, just utility name, state, zip
- why is the exact match on name + address better?

A few fields beyond the core address/name/year set may also help if they exist in the source tables:
- utility or respondent IDs
- EIN or other tax identifiers
- phone numbers
- service territory or state of incorporation
- website or contact metadata

Those can be added as exact-match or high-weight comparison columns once the basic utility/address linkage is done.